In [55]:
import pandas as pd
from pathlib import Path

In [56]:
raw_path = Path("data/raw")

In [57]:
dateien = list(raw_path.iterdir())

for datei in dateien:
    print(datei.name)

Kärnten_1900_1949_raw.json
Kärnten_1950_1989_raw.json
Kärnten_1990_2026_raw.json
Niederoesterreich_1900_1935_raw.json
Niederoesterreich_1936_1966_raw.json
Niederoesterreich_1967_1997_raw.json
Niederoesterreich_1998_2026_raw.json
Oberösterreich_1900_2026_raw.csv
parameter_metadaten.csv
stations_metadaten.csv


In [58]:
klima_dateien = [
    datei for datei in dateien
    if datei.name not in ["parameter_metadaten.csv", "stations_metadaten.csv"]
]

klima_dateien

[WindowsPath('data/raw/Kärnten_1900_1949_raw.json'),
 WindowsPath('data/raw/Kärnten_1950_1989_raw.json'),
 WindowsPath('data/raw/Kärnten_1990_2026_raw.json'),
 WindowsPath('data/raw/Niederoesterreich_1900_1935_raw.json'),
 WindowsPath('data/raw/Niederoesterreich_1936_1966_raw.json'),
 WindowsPath('data/raw/Niederoesterreich_1967_1997_raw.json'),
 WindowsPath('data/raw/Niederoesterreich_1998_2026_raw.json'),
 WindowsPath('data/raw/Oberösterreich_1900_2026_raw.csv')]

In [59]:
import json

test_json = raw_path / "Kärnten_1900_1949_raw.json"

with open(test_json, "r", encoding="utf-8") as f:
    json_data = json.load(f)

type(json_data)

dict

In [60]:
json_data.keys()

dict_keys(['media_type', 'type', 'version', 'timestamps', 'features'])

In [61]:
type(json_data["features"])

list

In [62]:
len(json_data["features"])

158

In [63]:
json_data["features"][0].keys()

dict_keys(['type', 'geometry', 'properties'])

In [64]:
json_data["features"][0]

{'type': 'Feature',
 'geometry': {'type': 'Point', 'coordinates': [47.049999, 12.8]},
 'properties': {'parameters': {'rr': {'name': 'Niederschlag Summe der 24h-Summen',
    'unit': 'mm',
    'data': [None,
     None,
     None,
     None,
     None,
     None,
     None,
     None,
     None,
     None,
     None,
     None,
     None,
     None,
     None,
     None,
     None,
     None,
     None,
     None,
     None,
     None,
     None,
     None,
     None,
     None,
     None,
     None,
     None,
     None,
     None,
     None,
     None,
     None,
     None,
     None,
     None,
     None,
     None,
     None,
     None,
     None,
     None,
     None,
     None,
     None,
     None,
     None,
     None,
     None,
     None,
     None,
     None,
     None,
     None,
     None,
     None,
     None,
     None,
     None,
     None,
     None,
     None,
     None,
     None,
     None,
     None,
     None,
     None,
     None,
     None,
     None,
     None,
  

In [65]:
json_data["features"][0]["properties"]["parameters"].keys()

dict_keys(['rr', 'sh_manu_max', 'shneu_manu', 'tage_festrrp', 'tage_frost', 'tage_schdecke', 'tlmax_mittel', 'tlmin_mittel', 'tl_mittel'])

In [66]:
json_data["features"][0]["properties"]["parameters"].keys()

dict_keys(['rr', 'sh_manu_max', 'shneu_manu', 'tage_festrrp', 'tage_frost', 'tage_schdecke', 'tlmax_mittel', 'tlmin_mittel', 'tl_mittel'])

In [67]:
def json_zu_dataframe(datei):
    with open(datei, "r", encoding="utf-8") as f:
        data = json.load(f)

    timestamps = data["timestamps"]
    rows = []

    for feature in data["features"]:
        station = feature["properties"]["station"]
        parameter = feature["properties"]["parameters"]

        for i, time in enumerate(timestamps):
            row = {
                "time": time,
                "station": station
            }

            for name, values in parameter.items():
                row[name] = values["data"][i]

            rows.append(row)

    return pd.DataFrame(rows)

In [68]:
test_kaernten = json_zu_dataframe(
    raw_path / "Kärnten_1900_1949_raw.json"
)

test_kaernten.head()

,time,station,rr,sh_manu_max,shneu_manu,tage_festrrp,tage_frost,tage_schdecke,tlmax_mittel,tlmin_mittel,tl_mittel
0,1900-01-01T00:00+00:00,15360,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,1900-02-01T00:00+00:00,15360,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,1900-03-01T00:00+00:00,15360,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,1900-04-01T00:00+00:00,15360,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,1900-05-01T00:00+00:00,15360,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [69]:
test_kaernten.shape

(94800, 11)

In [70]:
dataframes = []

for datei in klima_dateien:
    if datei.suffix == ".csv":
        df = pd.read_csv(datei)

    elif datei.suffix == ".json":
        df = json_zu_dataframe(datei)

    else:
        continue

    df["Quelldatei"] = datei.name
    dataframes.append(df)

klima_gesamt = pd.concat(dataframes, ignore_index=True)

In [71]:
klima_gesamt.shape

(847197, 13)

In [72]:
klima_gesamt["station"].nunique()

557

In [73]:
klima_gesamt.head()

,time,station,rr,sh_manu_max,shneu_manu,tage_festrrp,tage_frost,tage_schdecke,tlmax_mittel,tlmin_mittel,tl_mittel,Quelldatei,substation
0,1900-01-01T00:00+00:00,15360,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Kärnten_1900_1949_raw.json,NaN
1,1900-02-01T00:00+00:00,15360,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Kärnten_1900_1949_raw.json,NaN
2,1900-03-01T00:00+00:00,15360,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Kärnten_1900_1949_raw.json,NaN
3,1900-04-01T00:00+00:00,15360,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Kärnten_1900_1949_raw.json,NaN
4,1900-05-01T00:00+00:00,15360,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Kärnten_1900_1949_raw.json,NaN


In [74]:
klima_gesamt.info()

<class 'pandas.DataFrame'>
RangeIndex: 847197 entries, 0 to 847196
Data columns (total 13 columns):
 #   Column         Non-Null Count   Dtype  
---  ------         --------------   -----  
 0   time           847197 non-null  str    
 1   station        847197 non-null  int64  
 2   rr             200216 non-null  float64
 3   sh_manu_max    168676 non-null  float64
 4   shneu_manu     101213 non-null  float64
 5   tage_festrrp   164330 non-null  float64
 6   tage_frost     188084 non-null  float64
 7   tage_schdecke  168200 non-null  float64
 8   tlmax_mittel   191651 non-null  float64
 9   tlmin_mittel   191654 non-null  float64
 10  tl_mittel      199761 non-null  float64
 11  Quelldatei     847197 non-null  str    
 12  substation     23645 non-null   float64
dtypes: float64(10), int64(1), str(2)
memory usage: 84.0 MB


In [75]:
stations = pd.read_csv("data/raw/stations_metadaten.csv")

klima_gesamt = klima_gesamt.merge(
    stations[
        [
            "id",
            "Stationsname",
            "Höhe [m]",
            "Bundesland",
            "Länge [°E]",
            "Breite [°N]",
            "Startdatum",
            "Enddatum"
        ]
    ],
    left_on="station",
    right_on="id",
    how="left"
)

klima_gesamt.head()

,time,station,rr,sh_manu_max,shneu_manu,tage_festrrp,tage_frost,tage_schdecke,tlmax_mittel,tlmin_mittel,...,Quelldatei,substation,id,Stationsname,Höhe [m],Bundesland,Länge [°E],Breite [°N],Startdatum,Enddatum
0,1900-01-01T00:00+00:00,15360,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,Kärnten_1900_1949_raw.json,NaN,15360,Palik,1950.0,Kärnten,12.8,47.049999,1953-01-01 00:00:00+00:00,1977-10-01 00:00:00+00:00
1,1900-02-01T00:00+00:00,15360,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,Kärnten_1900_1949_raw.json,NaN,15360,Palik,1950.0,Kärnten,12.8,47.049999,1953-01-01 00:00:00+00:00,1977-10-01 00:00:00+00:00
2,1900-03-01T00:00+00:00,15360,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,Kärnten_1900_1949_raw.json,NaN,15360,Palik,1950.0,Kärnten,12.8,47.049999,1953-01-01 00:00:00+00:00,1977-10-01 00:00:00+00:00
3,1900-04-01T00:00+00:00,15360,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,Kärnten_1900_1949_raw.json,NaN,15360,Palik,1950.0,Kärnten,12.8,47.049999,1953-01-01 00:00:00+00:00,1977-10-01 00:00:00+00:00
4,1900-05-01T00:00+00:00,15360,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,Kärnten_1900_1949_raw.json,NaN,15360,Palik,1950.0,Kärnten,12.8,47.049999,1953-01-01 00:00:00+00:00,1977-10-01 00:00:00+00:00


In [76]:
klima_gesamt["Stationsname"].isna().sum()

np.int64(0)

In [77]:
klima_gesamt.groupby("Bundesland")["station"].nunique().sort_values(ascending=False)

Bundesland
Niederösterreich    252
Kärnten             158
Oberösterreich      147
Name: station, dtype: int64

In [78]:
klima_gesamt["time"] = pd.to_datetime(klima_gesamt["time"], utc=True)

klima_gesamt["time"].min(), klima_gesamt["time"].max()

(Timestamp('1900-01-01 00:00:00+0000', tz='UTC'),
 Timestamp('2026-09-01 00:00:00+0000', tz='UTC'))

In [79]:
datei_check = (
    klima_gesamt.groupby(["Quelldatei", "Bundesland"])["station"]
    .nunique()
    .reset_index()
)

datei_check

,Quelldatei,Bundesland,station
0,Kärnten_1900_1949_raw.json,Kärnten,158
1,Kärnten_1950_1989_raw.json,Kärnten,158
2,Kärnten_1990_2026_raw.json,Kärnten,158
3,Niederoesterreich_1900_1935_raw.json,Niederösterreich,252
4,Niederoesterreich_1936_1966_raw.json,Niederösterreich,252
5,Niederoesterreich_1967_1997_raw.json,Niederösterreich,252
6,Niederoesterreich_1998_2026_raw.json,Niederösterreich,252
7,Oberösterreich_1900_2026_raw.csv,Oberösterreich,147


In [80]:
datei_check = (
    klima_gesamt.groupby(["Quelldatei", "Bundesland"])["station"]
    .nunique()
    .reset_index()
)

datei_check

,Quelldatei,Bundesland,station
0,Kärnten_1900_1949_raw.json,Kärnten,158
1,Kärnten_1950_1989_raw.json,Kärnten,158
2,Kärnten_1990_2026_raw.json,Kärnten,158
3,Niederoesterreich_1900_1935_raw.json,Niederösterreich,252
4,Niederoesterreich_1936_1966_raw.json,Niederösterreich,252
5,Niederoesterreich_1967_1997_raw.json,Niederösterreich,252
6,Niederoesterreich_1998_2026_raw.json,Niederösterreich,252
7,Oberösterreich_1900_2026_raw.csv,Oberösterreich,147


In [83]:
for datei in [
    "Kärnten_1900_1949_raw.json",
    "Kärnten_1950_1989_raw.json",
    "Kärnten_1990_2026_raw.json"
]:
    df = json_zu_dataframe(raw_path / datei)
    df["time"] = pd.to_datetime(df["time"], utc=True)

    print(
        datei,
        "| Stationen:", df["station"].nunique(),
        "| Von:", df["time"].min(),
        "| Bis:", df["time"].max(),
        "| Zeilen:", len(df)
    )

Kärnten_1900_1949_raw.json | Stationen: 158 | Von: 1900-01-01 00:00:00+00:00 | Bis: 1949-12-01 00:00:00+00:00 | Zeilen: 94800
Kärnten_1950_1989_raw.json | Stationen: 158 | Von: 1950-01-01 00:00:00+00:00 | Bis: 1989-12-01 00:00:00+00:00 | Zeilen: 75840
Kärnten_1990_2026_raw.json | Stationen: 158 | Von: 1990-01-01 00:00:00+00:00 | Bis: 2026-09-01 00:00:00+00:00 | Zeilen: 69678


In [84]:
for datei in [
    "Niederoesterreich_1900_1935_raw.json",
    "Niederoesterreich_1936_1966_raw.json",
    "Niederoesterreich_1967_1997_raw.json",
    "Niederoesterreich_1998_2026_raw.json"
]:
    df = json_zu_dataframe(raw_path / datei)
    df["time"] = pd.to_datetime(df["time"], utc=True)

    print(
        datei,
        "| Stationen:", df["station"].nunique(),
        "| Von:", df["time"].min(),
        "| Bis:", df["time"].max(),
        "| Zeilen:", len(df)
    )

Niederoesterreich_1900_1935_raw.json | Stationen: 252 | Von: 1900-01-01 00:00:00+00:00 | Bis: 1935-12-01 00:00:00+00:00 | Zeilen: 108864
Niederoesterreich_1936_1966_raw.json | Stationen: 252 | Von: 1936-01-01 00:00:00+00:00 | Bis: 1966-12-01 00:00:00+00:00 | Zeilen: 93744
Niederoesterreich_1967_1997_raw.json | Stationen: 252 | Von: 1967-01-01 00:00:00+00:00 | Bis: 1997-12-01 00:00:00+00:00 | Zeilen: 93744
Niederoesterreich_1998_2026_raw.json | Stationen: 252 | Von: 1998-01-01 00:00:00+00:00 | Bis: 2026-09-01 00:00:00+00:00 | Zeilen: 86940


In [85]:
import pandas as pd
from pathlib import Path

# ============================================================
# 1. GRUNDEINSTELLUNGEN
# ============================================================

raw_path = Path("data/raw")

erwartete_bundeslaender = {
    "Burgenland",
    "Kärnten",
    "Niederösterreich",
    "Oberösterreich",
    "Salzburg",
    "Steiermark",
    "Tirol",
    "Vorarlberg",
    "Wien"
}

parameter = [
    "tl_mittel",
    "tlmax_mittel",
    "tlmin_mittel",
    "tage_frost",
    "shneu_manu",
    "tage_schdecke",
    "sh_manu_max",
    "rr",
    "tage_festrrp"
]

# Nur die eigentlichen Klima-Rohdateien laden
klima_dateien = sorted(raw_path.glob("*_raw.csv"))

print("========================================")
print("1. GEFUNDENE KLIMADATEIEN")
print("========================================")

for datei in klima_dateien:
    print(datei.name)

print(f"\nAnzahl Klimadateien: {len(klima_dateien)}")


# ============================================================
# 2. DATEIEN EINLESEN UND STRUKTUR PRÜFEN
# ============================================================

dataframes = []
datei_infos = []

for datei in klima_dateien:

    df = pd.read_csv(datei)

    # Datum umwandeln
    df["time"] = pd.to_datetime(df["time"], utc=True)

    # Quelldatei ergänzen
    df["Quelldatei"] = datei.name

    # Prüfen, welche erwarteten Parameter fehlen
    fehlende_parameter = [
        p for p in parameter
        if p not in df.columns
    ]

    datei_infos.append({
        "Datei": datei.name,
        "Zeilen": len(df),
        "Stationen": df["station"].nunique(),
        "Start": df["time"].min(),
        "Ende": df["time"].max(),
        "Fehlende Parameter": ", ".join(fehlende_parameter)
            if fehlende_parameter else "Keine"
    })

    dataframes.append(df)


datei_check = pd.DataFrame(datei_infos)

print("\n========================================")
print("2. DATEI-CHECK")
print("========================================")

display(datei_check)


# ============================================================
# 3. ALLE BUNDESLÄNDER ZUSAMMENFÜHREN
# ============================================================

klima_gesamt = pd.concat(
    dataframes,
    ignore_index=True
)

print("\n========================================")
print("3. GESAMTDATENSATZ")
print("========================================")

print("Zeilen:", len(klima_gesamt))
print("Spalten:", klima_gesamt.shape[1])
print("Eindeutige Stations-IDs:", klima_gesamt["station"].nunique())

print("\nZeitraum:")
print(
    klima_gesamt["time"].min(),
    "bis",
    klima_gesamt["time"].max()
)


# ============================================================
# 4. STATIONSMETADATEN LADEN
# ============================================================

stations = pd.read_csv(
    raw_path / "stations_metadaten.csv"
)

print("\n========================================")
print("4. STATIONSMETADATEN")
print("========================================")

print("Stationen in Metadaten:", stations["id"].nunique())


# ============================================================
# 5. METADATEN MIT KLIMADATEN VERBINDEN
# ============================================================

klima_gesamt = klima_gesamt.merge(
    stations[
        [
            "id",
            "Stationsname",
            "Höhe [m]",
            "Bundesland",
            "Länge [°E]",
            "Breite [°N]",
            "Startdatum",
            "Enddatum"
        ]
    ],
    left_on="station",
    right_on="id",
    how="left"
)

fehlende_station_metadata = klima_gesamt[
    "Stationsname"
].isna().sum()

print("\n========================================")
print("5. METADATEN-MERGE")
print("========================================")

print(
    "Zeilen ohne passende Stationsmetadaten:",
    fehlende_station_metadata
)


# ============================================================
# 6. BUNDESLÄNDER PRÜFEN
# ============================================================

bundesland_check = (
    klima_gesamt
    .groupby("Bundesland")["station"]
    .nunique()
    .sort_values(ascending=False)
)

print("\n========================================")
print("6. STATIONEN PRO BUNDESLAND")
print("========================================")

display(bundesland_check)

gefundene_bundeslaender = set(
    klima_gesamt["Bundesland"].dropna().unique()
)

fehlende_bundeslaender = (
    erwartete_bundeslaender
    - gefundene_bundeslaender
)

unerwartete_bundeslaender = (
    gefundene_bundeslaender
    - erwartete_bundeslaender
)

print("Fehlende Bundesländer:", fehlende_bundeslaender)
print("Unerwartete Bundesländer:", unerwartete_bundeslaender)


# ============================================================
# 7. PRÜFEN, OB JEDE DATEI DAS RICHTIGE BUNDESLAND ENTHÄLT
# ============================================================

quelle_bundesland_check = (
    klima_gesamt
    .groupby(["Quelldatei", "Bundesland"])["station"]
    .nunique()
    .reset_index()
)

print("\n========================================")
print("7. QUELLDATEI → BUNDESLAND")
print("========================================")

display(quelle_bundesland_check)


# ============================================================
# 8. FEHLENDE WERTE DER KLIMAPARAMETER
# ============================================================

missing_check = pd.DataFrame({
    "Fehlende Werte":
        klima_gesamt[parameter].isna().sum(),

    "Fehlend in %":
        (
            klima_gesamt[parameter]
            .isna()
            .mean()
            * 100
        ).round(2)
})

print("\n========================================")
print("8. MISSING VALUES")
print("========================================")

display(missing_check)


# ============================================================
# 9. EXAKTE DOPPELTE ZEILEN PRÜFEN
# ============================================================

exakte_dubletten = klima_gesamt.duplicated().sum()

print("\n========================================")
print("9. DUBLETTEN")
print("========================================")

print("Exakt doppelte Zeilen:", exakte_dubletten)


# ============================================================
# 10. STATION + MONAT MEHRFACH VORHANDEN?
# ============================================================

station_time_dubletten = (
    klima_gesamt
    .duplicated(
        subset=["station", "time"],
        keep=False
    )
    .sum()
)

print(
    "Zeilen mit mehrfach vorkommender "
    "Station-Zeit-Kombination:",
    station_time_dubletten
)


# ============================================================
# 11. DATENABDECKUNG JE BUNDESLAND
# ============================================================

abdeckung = (
    klima_gesamt
    .groupby("Bundesland")[parameter]
    .count()
)

print("\n========================================")
print("10. VORHANDENE MESSWERTE PRO BUNDESLAND")
print("========================================")

display(abdeckung)


# ============================================================
# 12. ABSCHLUSS-CHECK
# ============================================================

print("\n========================================")
print("ABSCHLUSS")
print("========================================")

if (
    len(klima_dateien) == 9
    and fehlende_station_metadata == 0
    and len(fehlende_bundeslaender) == 0
    and len(unerwartete_bundeslaender) == 0
):

    print("✅ Grundstruktur der Rohdaten sieht korrekt aus.")

else:

    print("⚠️ Es gibt noch Punkte, die geprüft werden sollten.")

1. GEFUNDENE KLIMADATEIEN
Burgenland_1900_2026_raw.csv
Kärnten_1900_2026_raw.csv
Niederösterreich_1900_2026_raw.csv
Oberösterreich_1900_2026_raw.csv
Salzburg_1900_2026_raw.csv
Steiermark_1900_2026_raw.csv
Tirol_1900_2026_raw.csv
Vorarlberg_1900_2026_raw.csv
Wien_1900_2026_raw.csv

Anzahl Klimadateien: 9

2. DATEI-CHECK


,Datei,Zeilen,Stationen,Start,Ende,Fehlende Parameter
0,Burgenland_1900_2026_raw.csv,85176,56,1900-01-01 00:00:00+00:00,2026-09-01 00:00:00+00:00,Keine
1,Kärnten_1900_2026_raw.csv,240318,158,1900-01-01 00:00:00+00:00,2026-09-01 00:00:00+00:00,Keine
2,Niederösterreich_1900_2026_raw.csv,383292,252,1900-01-01 00:00:00+00:00,2026-09-01 00:00:00+00:00,Keine
3,Oberösterreich_1900_2026_raw.csv,223587,147,1900-01-01 00:00:00+00:00,2026-09-01 00:00:00+00:00,Keine
4,Salzburg_1900_2026_raw.csv,187083,123,1900-01-01 00:00:00+00:00,2026-09-01 00:00:00+00:00,Keine
5,Steiermark_1900_2026_raw.csv,228150,150,1900-01-01 00:00:00+00:00,2026-09-01 00:00:00+00:00,Keine
6,Tirol_1900_2026_raw.csv,267696,176,1900-01-01 00:00:00+00:00,2026-09-01 00:00:00+00:00,Keine
7,Vorarlberg_1900_2026_raw.csv,94302,62,1900-01-01 00:00:00+00:00,2026-09-01 00:00:00+00:00,Keine
8,Wien_1900_2026_raw.csv,27378,18,1900-01-01 00:00:00+00:00,2026-09-01 00:00:00+00:00,Keine



3. GESAMTDATENSATZ
Zeilen: 1736982
Spalten: 13
Eindeutige Stations-IDs: 1142

Zeitraum:
1900-01-01 00:00:00+00:00 bis 2026-09-01 00:00:00+00:00

4. STATIONSMETADATEN
Stationen in Metadaten: 1144

5. METADATEN-MERGE
Zeilen ohne passende Stationsmetadaten: 0

6. STATIONEN PRO BUNDESLAND


Bundesland
Niederösterreich    252
Tirol               176
Kärnten             158
Steiermark          150
Oberösterreich      147
Salzburg            123
Vorarlberg           62
Burgenland           56
Wien                 18
Name: station, dtype: int64

Fehlende Bundesländer: set()
Unerwartete Bundesländer: set()

7. QUELLDATEI → BUNDESLAND


...


8. MISSING VALUES


,Fehlende Werte,Fehlend in %
tl_mittel,1308266,75.32
tlmax_mittel,1328041,76.46
tlmin_mittel,1328001,76.45
tage_frost,1335242,76.87
shneu_manu,1507872,86.81
tage_schdecke,1377404,79.30
sh_manu_max,1376920,79.27
rr,1302340,74.98
tage_festrrp,1386272,79.81



9. DUBLETTEN
Exakt doppelte Zeilen: 0
Zeilen mit mehrfach vorkommender Station-Zeit-Kombination: 0

10. VORHANDENE MESSWERTE PRO BUNDESLAND


,tl_mittel,tlmax_mittel,tlmin_mittel,tage_frost,shneu_manu,tage_schdecke,sh_manu_max,rr,tage_festrrp
Bundesland,,,,,,,,,
Burgenland,18005,17573,17555,17415,10158,15150,15149,17922,15581
Kärnten,56527,54591,54623,53327,28629,48254,48254,55858,47966
Niederösterreich,84503,82389,82400,80613,39690,68973,68969,84618,68251
Oberösterreich,58731,54671,54631,54144,32894,50973,51453,59740,48113
Salzburg,43923,42752,42672,42348,25901,38058,38062,45283,38419
Steiermark,70559,65208,65368,62553,40056,58200,58198,69767,54165
Tirol,67651,64607,64579,64278,36277,56856,56863,70601,56673
Vorarlberg,19942,17921,17933,17843,11874,17564,17564,21672,15180
Wien,8875,9229,9220,9219,3631,5550,5550,9181,6362



ABSCHLUSS
✅ Grundstruktur der Rohdaten sieht korrekt aus.


In [86]:
klima_gesamt[["station", "Stationsname", "Höhe [m]", "Bundesland"]].drop_duplicates()["Höhe [m]"].describe()

count    1142.000000
mean      714.159457
std       500.232254
min       116.000000
25%       384.750000
50%       578.000000
75%       908.700000
max      3437.000000
Name: Höhe [m], dtype: float64

In [87]:
stationen = klima_gesamt[
    ["station", "Stationsname", "Höhe [m]", "Bundesland"]
].drop_duplicates()

stationen.sort_values("Höhe [m]").head(10)

,station,Stationsname,Höhe [m],Bundesland
16,7912,Podersdorf Strandbad,116.0,Burgenland
50,65,Neusiedl am See,117.0,Burgenland
9,7890,Neusiedl am See,117.0,Burgenland
7,7818,Illmitz,117.0,Burgenland
55,218,Andau,117.0,Burgenland
20,7956,Andau,117.0,Burgenland
19,7955,Andau,117.5,Burgenland
18,7950,Andau,118.0,Burgenland
25,10900,Apetlon,118.0,Burgenland
8,7821,Rust am See,122.0,Burgenland


In [88]:
stationen.sort_values("Höhe [m]", ascending=False).head(10)

,station,Stationsname,Höhe [m],Bundesland
1347738,17320,Brunnenkogel,3437.0,Tirol
932443,15410,Sonnblick,3109.0,Salzburg
932444,15411,Sonnblick,3109.0,Salzburg
932495,213,Sonnblick,3109.0,Salzburg
932445,15412,Sonnblick Fensterhütte,3105.0,Salzburg
932446,15413,Sonnblick Nord alt,3105.0,Salzburg
1347775,184,Pitztaler Gletscher,2863.9,Tirol
1347737,17315,Pitztaler Gletscher,2863.9,Tirol
1347736,17310,Pitztaler Gletscher,2850.0,Tirol
1347672,14308,Valluga,2805.0,Tirol


In [89]:
bins = [0, 500, 1000, 1500, float("inf")]
labels = [
    "unter 500 m",
    "500–999 m",
    "1000–1499 m",
    "ab 1500 m"
]

klima_gesamt["Höhenklasse"] = pd.cut(
    klima_gesamt["Höhe [m]"],
    bins=bins,
    labels=labels,
    right=False
)

stationen_hoehe = klima_gesamt[
    ["station", "Stationsname", "Höhe [m]", "Bundesland", "Höhenklasse"]
].drop_duplicates()

stationen_hoehe["Höhenklasse"].value_counts().sort_index()

Höhenklasse
unter 500 m    460
500–999 m      445
1000–1499 m    156
ab 1500 m       81
Name: count, dtype: int64

In [90]:
klima_gesamt["Jahr"] = klima_gesamt["time"].dt.year
klima_gesamt["Monat"] = klima_gesamt["time"].dt.month

klima_gesamt["Jahreszeit"] = klima_gesamt["Monat"].map({
    12: "Winter", 1: "Winter", 2: "Winter",
    3: "Frühling", 4: "Frühling", 5: "Frühling",
    6: "Sommer", 7: "Sommer", 8: "Sommer",
    9: "Herbst", 10: "Herbst", 11: "Herbst"
})

klima_gesamt[["time", "Jahr", "Monat", "Jahreszeit"]].head()

,time,Jahr,Monat,Jahreszeit
0,1900-01-01 00:00:00+00:00,1900,1,Winter
1,1900-01-01 00:00:00+00:00,1900,1,Winter
2,1900-01-01 00:00:00+00:00,1900,1,Winter
3,1900-01-01 00:00:00+00:00,1900,1,Winter
4,1900-01-01 00:00:00+00:00,1900,1,Winter


In [91]:
parameter = [
    "tl_mittel",
    "tlmax_mittel",
    "tlmin_mittel",
    "tage_frost",
    "shneu_manu",
    "tage_schdecke",
    "sh_manu_max",
    "rr",
    "tage_festrrp"
]

qualitaet_station = klima_gesamt.groupby(
    ["station", "Stationsname", "Bundesland", "Höhe [m]", "Höhenklasse"]
)[parameter].count()

qualitaet_station.head()

,,,,,tl_mittel,tlmax_mittel,tlmin_mittel,tage_frost,shneu_manu,tage_schdecke,sh_manu_max,rr,tage_festrrp
station,Stationsname,Bundesland,Höhe [m],Höhenklasse,,,,,,,,,
1,Aflenz,Steiermark,783.2,500–999 m,713,713,713,713,662,707,707,712,705
2,Aigen im Ennstal,Steiermark,641.0,500–999 m,835,835,835,834,570,834,834,834,831
3,Allentsteig,Niederösterreich,598.8,500–999 m,515,515,515,515,515,515,515,513,512
4,Amstetten,Niederösterreich,266.0,unter 500 m,1029,1028,1027,1026,92,568,568,1024,629
5,Bad Aussee,Steiermark,743.1,500–999 m,830,830,830,829,569,831,831,829,826


In [92]:
zeitraum_station = (
    klima_gesamt
    .dropna(subset=["tl_mittel"])
    .groupby(
        ["station", "Stationsname", "Bundesland", "Höhe [m]", "Höhenklasse"]
    )["time"]
    .agg(["min", "max", "count"])
    .reset_index()
)

zeitraum_station.sort_values("count", ascending=False).head(20)

,station,Stationsname,Bundesland,Höhe [m],Höhenklasse,min,max,count
38,39,Innsbruck Universität,Tirol,578.0,500–999 m,1900-01-01 00:00:00+00:00,2026-08-01 00:00:00+00:00,1520
102,105,Wien Hohe Warte,Wien,198.0,unter 500 m,1900-01-01 00:00:00+00:00,2026-08-01 00:00:00+00:00,1520
206,213,Sonnblick,Salzburg,3109.0,ab 1500 m,1900-01-01 00:00:00+00:00,2026-08-01 00:00:00+00:00,1520
197,204,Kremsmünster,Oberösterreich,382.0,unter 500 m,1900-01-01 00:00:00+00:00,2026-08-01 00:00:00+00:00,1519
113,116,Deutschlandsberg,Steiermark,354.0,unter 500 m,1900-01-01 00:00:00+00:00,2026-08-01 00:00:00+00:00,1518
29,30,Graz Universität/Heinrichstraße,Steiermark,366.0,unter 500 m,1900-01-01 00:00:00+00:00,2026-08-01 00:00:00+00:00,1513
25,26,Freistadt,Oberösterreich,539.0,500–999 m,1900-01-01 00:00:00+00:00,2026-08-01 00:00:00+00:00,1511
23,24,Feldkirch,Vorarlberg,438.4,unter 500 m,1900-01-01 00:00:00+00:00,2026-08-01 00:00:00+00:00,1507
27,28,Galtür,Tirol,1587.0,ab 1500 m,1901-01-01 00:00:00+00:00,2026-08-01 00:00:00+00:00,1504
151,157,Kollerschlag,Oberösterreich,714.0,500–999 m,1900-01-01 00:00:00+00:00,2026-08-01 00:00:00+00:00,1499


In [93]:
stationen_hoehe["Höhenklasse"].value_counts().sort_index()

Höhenklasse
unter 500 m    460
500–999 m      445
1000–1499 m    156
ab 1500 m       81
Name: count, dtype: int64

In [94]:
jahresdaten = klima_gesamt.groupby(
    [
        "station",
        "Stationsname",
        "Bundesland",
        "Höhe [m]",
        "Höhenklasse",
        "Jahr"
    ]
).agg(
    tl_mittel=("tl_mittel", "mean"),
    tlmax_mittel=("tlmax_mittel", "mean"),
    tlmin_mittel=("tlmin_mittel", "mean"),

    tage_frost=("tage_frost", lambda x: x.sum(min_count=10)),
    shneu_manu=("shneu_manu", lambda x: x.sum(min_count=10)),
    tage_schdecke=("tage_schdecke", lambda x: x.sum(min_count=10)),

    sh_manu_max=("sh_manu_max", "max"),

    rr=("rr", lambda x: x.sum(min_count=10))
).reset_index()

jahresdaten.head()

,station,Stationsname,Bundesland,Höhe [m],Höhenklasse,Jahr,tl_mittel,tlmax_mittel,tlmin_mittel,tage_frost,shneu_manu,tage_schdecke,sh_manu_max,rr
0,1,Aflenz,Steiermark,783.2,500–999 m,1900,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,1,Aflenz,Steiermark,783.2,500–999 m,1901,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,1,Aflenz,Steiermark,783.2,500–999 m,1902,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,1,Aflenz,Steiermark,783.2,500–999 m,1903,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,1,Aflenz,Steiermark,783.2,500–999 m,1904,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [95]:
jahresdaten.shape
jahresdaten.isna().sum()
jahresdaten["Jahr"].min(), jahresdaten["Jahr"].max()

(np.int32(1900), np.int32(2026))

In [96]:
def mittelwert_mit_mindestmonaten(x):
    return x.mean() if x.count() >= 10 else float("nan")

def summe_mit_mindestmonaten(x):
    return x.sum() if x.count() >= 10 else float("nan")

def maximum_mit_mindestmonaten(x):
    return x.max() if x.count() >= 10 else float("nan")


jahresdaten = klima_gesamt.groupby(
    [
        "station",
        "Stationsname",
        "Bundesland",
        "Höhe [m]",
        "Höhenklasse",
        "Jahr"
    ],
    observed=True
).agg(
    tl_mittel=("tl_mittel", mittelwert_mit_mindestmonaten),
    tlmax_mittel=("tlmax_mittel", mittelwert_mit_mindestmonaten),
    tlmin_mittel=("tlmin_mittel", mittelwert_mit_mindestmonaten),

    tage_frost=("tage_frost", summe_mit_mindestmonaten),
    shneu_manu=("shneu_manu", summe_mit_mindestmonaten),
    tage_schdecke=("tage_schdecke", summe_mit_mindestmonaten),

    sh_manu_max=("sh_manu_max", maximum_mit_mindestmonaten),

    rr=("rr", summe_mit_mindestmonaten)
).reset_index()

In [97]:
messwerte = [
    "tl_mittel",
    "tlmax_mittel",
    "tlmin_mittel",
    "tage_frost",
    "shneu_manu",
    "tage_schdecke",
    "sh_manu_max",
    "rr"
]

jahresdaten = jahresdaten.dropna(
    subset=messwerte,
    how="all"
)

In [98]:
print("Form:", jahresdaten.shape)
print("Zeitraum:", jahresdaten["Jahr"].min(), "-", jahresdaten["Jahr"].max())

jahresdaten.isna().sum()

jahresdaten.head(10)

Form: (36098, 14)
Zeitraum: 1900 - 2025


,station,Stationsname,Bundesland,Höhe [m],Höhenklasse,Jahr,tl_mittel,tlmax_mittel,tlmin_mittel,tage_frost,shneu_manu,tage_schdecke,sh_manu_max,rr
68,1,Aflenz,Steiermark,783.2,500–999 m,1968,6.333333,11.350000,2.116667,145.0,NaN,98.0,60.0,837.0
69,1,Aflenz,Steiermark,783.2,500–999 m,1969,6.166667,11.225000,1.833333,162.0,NaN,112.0,47.0,760.0
70,1,Aflenz,Steiermark,783.2,500–999 m,1970,5.816667,10.758333,1.791667,166.0,NaN,104.0,85.0,1190.0
71,1,Aflenz,Steiermark,783.2,500–999 m,1971,6.033333,11.941667,1.791667,142.0,95.0,70.0,26.0,677.0
72,1,Aflenz,Steiermark,783.2,500–999 m,1972,5.591667,10.916667,1.825000,144.0,85.0,45.0,28.0,1042.0
73,1,Aflenz,Steiermark,783.2,500–999 m,1973,5.625000,11.175000,1.558333,171.0,210.0,117.0,75.0,891.0
74,1,Aflenz,Steiermark,783.2,500–999 m,1974,6.258333,11.475000,2.433333,147.0,220.0,106.0,50.0,916.0
75,1,Aflenz,Steiermark,783.2,500–999 m,1975,6.275000,11.833333,2.366667,148.0,125.0,73.0,35.0,926.0
76,1,Aflenz,Steiermark,783.2,500–999 m,1976,5.850000,11.400000,1.791667,143.0,189.0,109.0,55.0,895.0
77,1,Aflenz,Steiermark,783.2,500–999 m,1977,6.441667,12.050000,2.291667,136.0,109.0,90.0,50.0,841.0


In [99]:
# Monatsdaten exportieren
klima_gesamt.to_csv(
    "data/processed/climate_monthly.csv",
    index=False
)

# Jahresdaten exportieren
jahresdaten.to_csv(
    "data/processed/climate_yearly.csv",
    index=False
)

In [100]:
print("Monatsdaten:", klima_gesamt.shape)
print("Jahresdaten:", jahresdaten.shape)

Monatsdaten: (1736982, 25)
Jahresdaten: (36098, 14)
